<a href="https://colab.research.google.com/github/LizaHam123/sentiment-analysis-algerie-telecom/blob/main/camembert_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 # ============================================================================
# ÉTAPE 1: INSTALLATION ET IMPORTS
# ============================================================================

!pip install transformers torch scikit-learn pandas numpy matplotlib seaborn plotly
!pip install datasets accelerate

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import torch

import warnings
warnings.filterwarnings('ignore')

print(" Tous les packages installés")

 Tous les packages installés


In [ ]:
# ============================================================================
# ÉTAPE 2: CHARGEMENT ET PRÉPARATION DES DONNÉES
# ============================================================================

# Charger le fichier des commentaires nettoyés
from google.colab import files

print(" Uploadez votre fichier commentaires_nettoyes.csv:")
uploaded = files.upload()

# Lire le fichier CSV
df = pd.read_csv('commentaires_nettoyes.csv')

print(f" Données chargées: {len(df)} commentaires")
print(f" Colonnes: {list(df.columns)}")
print("\n Aperçu des données:")
print(df.head())

# Vérifier la distribution des sentiments
print(f"\n Distribution des sentiments:")
print(df['sentiment'].value_counts())

 Uploadez votre fichier commentaires_nettoyes.csv:


Saving commentaires_nettoyes.csv to commentaires_nettoyes (1).csv
 Données chargées: 9238 commentaires
 Colonnes: ['commentaire', 'commentaire_nettoye', 'commentaire_spacy', 'lemmes', 'tokens_utiles', 'sentiment', 'probleme']

 Aperçu des données:
                                         commentaire  \
0                       Plantez et vous récolterez.,   
1  Bravo et merci à tous les employés d’Algérie T...   
2  Votre entreprise ne pense plus qu’aux abonnés ...   
3  Ce qui nous réjouit, c’est le passage de Idoom...   
4  On demande juste à être raccordés à la nouvell...   

                                 commentaire_nettoye  \
0                        Plantez et vous récolterez.   
1  Bravo et merci à tous les employés d’Algérie T...   
2  Votre entreprise ne pense plus qu’aux abonnés ...   
3  Ce qui nous réjouit, c’est le passage de Idoom...   
4  On demande juste à être raccordés à la nouvell...   

                                   commentaire_spacy  \
0                     

In [ ]:
# ============================================================================
# ÉTAPE 3: PRÉPARATION DES DONNÉES POUR L'ENTRAÎNEMENT
# ============================================================================

# Nettoyer les données
df = df.dropna(subset=['commentaire', 'sentiment'])

# Encoder les labels
label_encoder = LabelEncoder()
df['sentiment_encoded'] = label_encoder.fit_transform(df['sentiment'])

# Mapping des labels
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(f"\n Mapping des labels: {label_mapping}")

# Division train/test
X_train, X_test, y_train, y_test = train_test_split(
    df['commentaire'],
    df['sentiment_encoded'],
    test_size=0.2,
    random_state=42,
    stratify=df['sentiment_encoded']
)

print(f"\n Données d'entraînement: {len(X_train)} commentaires")
print(f" Données de test: {len(X_test)} commentaires")


 Mapping des labels: {'neutre': np.int64(0), 'négatif': np.int64(1), 'positif': np.int64(2)}

 Données d'entraînement: 7390 commentaires
 Données de test: 1848 commentaires


In [ ]:
# ============================================================================
# ÉTAPE 4: MODÈLE CAMEMBERT POUR ANALYSE DE SENTIMENT
# ============================================================================

# Charger le modèle CamemBERT pré-entraîné
model_name = "camembert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Préparer les données pour le fine-tuning
def tokenize_data(texts, labels, tokenizer, max_length=128):
    """Tokenise les textes pour l'entraînement"""
    encodings = tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )

    dataset = []
    for i in range(len(texts)):
        dataset.append({
            'input_ids': encodings['input_ids'][i],
            'attention_mask': encodings['attention_mask'][i],
            'labels': torch.tensor(labels.iloc[i], dtype=torch.long)
        })

    return dataset

# Tokeniser les données
print(" Tokenisation des données...")
train_dataset = tokenize_data(X_train, y_train, tokenizer)
test_dataset = tokenize_data(X_test, y_test, tokenizer)

print(" Données tokenisées")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

 Tokenisation des données...
 Données tokenisées


In [ ]:
# ============================================================================
# ÉTAPE 5: ENTRAÎNEMENT DU MODÈLE (SANS WANDB)
# ============================================================================

import os
os.environ["WANDB_DISABLED"] = "true"

from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

class SentimentDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

# Créer les datasets
train_dataset_torch = SentimentDataset(train_dataset)
test_dataset_torch = SentimentDataset(test_dataset)

# Charger le modèle pour classification
num_labels = len(label_encoder.classes_)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

# Configuration d'entraînement (SANS WANDB)
training_args = TrainingArguments(
    output_dir='./sentiment_model',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to=None,
)

# Créer le trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_torch,
    eval_dataset=test_dataset_torch,
)

# Entraîner le modèle
print(" Début de l'entraînement...")
trainer.train()
print(" Entraînement terminé")

# Sauvegarder le modèle
trainer.save_model('./sentiment_model_final')
tokenizer.save_pretrained('./sentiment_model_final')

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


 Début de l'entraînement...


Epoch,Training Loss,Validation Loss
1,0.426000,0.528248


 Entraînement terminé


('./sentiment_model_final/tokenizer_config.json',
 './sentiment_model_final/special_tokens_map.json',
 './sentiment_model_final/sentencepiece.bpe.model',
 './sentiment_model_final/added_tokens.json',
 './sentiment_model_final/tokenizer.json')

In [ ]:
# ============================================================================
# ÉTAPE 6: ÉVALUATION DU MODÈLE
# ============================================================================

# Prédictions sur le test set
predictions = trainer.predict(test_dataset_torch)
y_pred = np.argmax(predictions.predictions, axis=1)

# Métriques
accuracy = accuracy_score(y_test, y_pred)
print(f"\n Précision du modèle: {accuracy:.3f}")

# Rapport de classification
class_report = classification_report(
    y_test, y_pred,
    target_names=label_encoder.classes_,
    output_dict=True
)

print(f"\n Rapport de classification:")
for sentiment, metrics in class_report.items():
    if sentiment in ['positif', 'negatif', 'neutre']:
        print(f"{sentiment}: Précision={metrics['precision']:.3f}, Rappel={metrics['recall']:.3f}")


 Précision du modèle: 0.802

 Rapport de classification:
neutre: Précision=0.716, Rappel=0.553
positif: Précision=0.822, Rappel=0.826
